In [1]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd
import numpy as np

# Data Cleaning and Preprocessing

In [2]:
df = pd.read_csv("pima_data - Copy.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,192,66,35,64,31.3,1.668,42,1
1,3,122,56,44,269,42.1,0.562,41,0
2,12,126,99,48,163,18.7,0.810,46,1
3,14,108,103,27,290,26.6,1.587,60,1
4,10,178,74,48,186,34.6,1.539,41,1


In [3]:
# No null values, all numerical types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.shape

(768, 9)

In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,6.875000,132.734375,78.796875,33.065104,154.665365,34.554818,1.260259,50.032552,0.769531
std,4.364429,37.549234,23.548344,15.028818,81.667825,9.019027,0.702278,16.981629,0.421407
min,0.000000,70.000000,40.000000,7.000000,15.000000,18.100000,0.100000,21.000000,0.000000
25%,3.000000,102.000000,59.000000,20.000000,82.000000,27.250000,0.642000,36.000000,1.000000
50%,7.000000,131.000000,77.000000,32.000000,155.000000,34.950000,1.234000,49.000000,1.000000
75%,11.000000,166.000000,99.250000,46.000000,224.000000,42.100000,1.883500,65.000000,1.000000
max,14.000000,199.000000,119.000000,59.000000,299.000000,49.900000,2.494000,79.000000,1.000000


In [6]:
# Checking for invalid zeros in any columns using 
# Code from ChatGPT, Prompt: "How to check for zeros in a pandas dataframe columns?"
# zeros in Pregnancies and Outcome columns make sense so I'll keep them
(df == 0).any()

Pregnancies                  True
Glucose                     False
BloodPressure               False
SkinThickness               False
Insulin                     False
BMI                         False
DiabetesPedigreeFunction    False
Age                         False
Outcome                      True
dtype: bool

In [7]:
X = df.drop(["Outcome"], axis=1)
y = df["Outcome"]

X.shape, y.shape

((768, 8), (768,))

In [8]:
df.corr()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
Pregnancies,1.000000,0.035948,-0.047197,0.054090,0.054655,0.010296,-0.018967,-0.015566,0.328125
Glucose,0.035948,1.000000,-0.045341,0.018130,0.033689,0.039485,-0.014549,0.027108,0.301894
BloodPressure,-0.047197,-0.045341,1.000000,0.001117,-0.075879,0.089671,-0.032142,-0.012630,-0.005512
SkinThickness,0.054090,0.018130,0.001117,1.000000,-0.009482,0.004758,0.076386,-0.000586,-0.000510
Insulin,0.054655,0.033689,-0.075879,-0.009482,1.000000,-0.017999,0.062219,0.036972,-0.003418
BMI,0.010296,0.039485,0.089671,0.004758,-0.017999,1.000000,-0.032092,0.021476,0.388904
DiabetesPedigreeFunction,-0.018967,-0.014549,-0.032142,0.076386,0.062219,-0.032092,1.000000,0.034891,0.016674
Age,-0.015566,0.027108,-0.012630,-0.000586,0.036972,0.021476,0.034891,1.000000,0.348303
Outcome,0.328125,0.301894,-0.005512,-0.000510,-0.003418,0.388904,0.016674,0.348303,1.000000


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((614, 8), (614,), (154, 8), (154,))

### Scaling the features using pipelines

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled.shape, X_test_scaled.shape

((614, 8), (154, 8))

In [11]:
rf_model = RandomForestClassifier(random_state=42)

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, random_state=42))
])

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True, random_state=42))
])

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

### Defining hyperparameter grids

In [12]:
# Note: Use 'model__C' instead of 'C' because the models are nested inside a Pipeline.
# The 'model__' prefix tells the grid search to look specifically at the step named "model". 
# (Source: AI Gemini)
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15]
}
lr_param_grid = {
    'model__C': [0.1, 1, 10]
}
svm_param_grid = {
    'model__kernel': ['linear', 'rbf'],
    'model__C': [0.1, 1, 10],
    'model__gamma': [0.01, 0.1, 1]
}
knn_param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11]
}

# Running Grid search CV for each model

### A. Random Forest

In [13]:
# cv=5, 5 fold cross-validation
rf_gs = GridSearchCV(
rf_model,
rf_param_grid,
cv=5,
scoring='accuracy'
)
rf_gs.fit(X_train, y_train)
print("Random Forest best params:", rf_gs.best_params_)
# round: round numbers in python - below is round to 3:
print("Random Forest best CV score:", round(rf_gs.best_score_, 3))

Random Forest best params: {'max_depth': 10, 'n_estimators': 200}
Random Forest best CV score: 0.977


### B. Logistic Regression

In [14]:
lr_gs = GridSearchCV(
lr_pipeline,
lr_param_grid,
cv=5,
scoring='accuracy'
)
lr_gs.fit(X_train, y_train)
print("Logistic Regression best params:", lr_gs.best_params_)
print("Logistic Regression best CV score:", round(lr_gs.best_score_, 3))

Logistic Regression best params: {'model__C': 0.1}
Logistic Regression best CV score: 0.886


### C. Support Vector Machines

In [15]:
svm_gs = GridSearchCV(
svm_pipeline,
svm_param_grid,
cv=5,
scoring='accuracy'
)
svm_gs.fit(X_train, y_train)
print("SVM best params:", svm_gs.best_params_)
print("SVM best CV score:", round(svm_gs.best_score_, 3))

SVM best params: {'model__C': 10, 'model__gamma': 0.1, 'model__kernel': 'rbf'}
SVM best CV score: 0.886


### D. K Nearest Neighbours

In [16]:
knn_gs = GridSearchCV(
knn_pipeline,
knn_param_grid,
cv=5,
scoring='accuracy'
)
knn_gs.fit(X_train, y_train)
print("KNN best params:", knn_gs.best_params_)
print("KNN best CV score:", round(knn_gs.best_score_, 3))

KNN best params: {'model__n_neighbors': 11}
KNN best CV score: 0.876


In [17]:
models_full = {
    "Random Forest": rf_gs.best_estimator_,
    "Logistic Regression": lr_gs.best_estimator_,
    "SVM": svm_gs.best_estimator_,
    "KNN": knn_gs.best_estimator_
}

for name, model in models_full.items():
    print(f"\n{name} (All Features)")
    print("-" * 40)
    

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Random Forest (All Features)
----------------------------------------
Accuracy: 0.9935064935064936
Confusion Matrix:
[[ 34   1]
 [  0 119]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        35
           1       0.99      1.00      1.00       119

    accuracy                           0.99       154
   macro avg       1.00      0.99      0.99       154
weighted avg       0.99      0.99      0.99       154


Logistic Regression (All Features)
----------------------------------------
Accuracy: 0.935064935064935
Confusion Matrix:
[[ 25  10]
 [  0 119]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.71      0.83        35
           1       0.92      1.00      0.96       119

    accuracy                           0.94       154
   macro avg       0.96      0.86      0.90       154
weighted avg       0.94      0.94      0.93       154


SVM (All F

# Training models using SelectKBest Features

In [18]:
selector = SelectKBest(score_func=f_classif, k=7)

X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)

In [19]:
rf_gs_kbest = GridSearchCV(
    rf_model,
    rf_param_grid,
    cv=5,
    scoring='accuracy'
)

rf_gs_kbest.fit(X_train_kbest, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
             param_grid={'max_depth': [5, 10, 15],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [20]:
lr_gs_kbest = GridSearchCV(
    lr_pipeline,
    lr_param_grid,
    cv=5,
    scoring='accuracy'
)

lr_gs_kbest.fit(X_train_kbest, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=10000,
                                                           random_state=42))]),
             param_grid={'model__C': [0.1, 1, 10]}, scoring='accuracy')

In [21]:
svm_gs_kbest = GridSearchCV(
    svm_pipeline,
    svm_param_grid,
    cv=5,
    scoring='accuracy'
)

svm_gs_kbest.fit(X_train_kbest, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        SVC(probability=True,
                                            random_state=42))]),
             param_grid={'model__C': [0.1, 1, 10],
                         'model__gamma': [0.01, 0.1, 1],
                         'model__kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [22]:
knn_gs_kbest = GridSearchCV(
    knn_pipeline,
    knn_param_grid,
    cv=5,
    scoring='accuracy'
)

knn_gs_kbest.fit(X_train_kbest, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', KNeighborsClassifier())]),
             param_grid={'model__n_neighbors': [3, 5, 7, 9, 11]},
             scoring='accuracy')

In [23]:
models_kbest = {
    "Random Forest": rf_gs_kbest.best_estimator_,
    "Logistic Regression": lr_gs_kbest.best_estimator_,
    "SVM": svm_gs_kbest.best_estimator_,
    "KNN": knn_gs_kbest.best_estimator_
}

for name, model in models_kbest.items():
    print(f"\n{name} (SelectKBest)")
    print("-" * 40)
    
    y_pred = model.predict(X_test_kbest)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Random Forest (SelectKBest)
----------------------------------------
Accuracy: 1.0
Confusion Matrix:
[[ 35   0]
 [  0 119]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        35
           1       1.00      1.00      1.00       119

    accuracy                           1.00       154
   macro avg       1.00      1.00      1.00       154
weighted avg       1.00      1.00      1.00       154


Logistic Regression (SelectKBest)
----------------------------------------
Accuracy: 0.948051948051948
Confusion Matrix:
[[ 27   8]
 [  0 119]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.77      0.87        35
           1       0.94      1.00      0.97       119

    accuracy                           0.95       154
   macro avg       0.97      0.89      0.92       154
weighted avg       0.95      0.95      0.95       154


SVM (SelectKBest)
---------

# Comparing Random Forest Feature Importance and SelectKBest

In [25]:
rf_best = rf_gs.best_estimator_

importances = rf_best.feature_importances_

rf_features = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print(rf_features)

                    Feature  Importance
5                       BMI    0.236903
7                       Age    0.195924
0               Pregnancies    0.192917
1                   Glucose    0.175307
2             BloodPressure    0.059090
6  DiabetesPedigreeFunction    0.049401
4                   Insulin    0.049202
3             SkinThickness    0.041255


In [26]:
top_rf = rf_features["Feature"].head(7)
print("Top features from Random Forest:")
print(top_rf)

Top features from Random Forest:
5                         BMI
7                         Age
0                 Pregnancies
1                     Glucose
2               BloodPressure
6    DiabetesPedigreeFunction
4                     Insulin
Name: Feature, dtype: object


In [27]:
scores = selector.scores_

kbest_features = pd.DataFrame({
    "Feature": X.columns,
    "Score": scores
}).sort_values("Score", ascending=False)

print(kbest_features)

                    Feature       Score
5                       BMI  102.307343
7                       Age   75.877128
0               Pregnancies   75.157588
1                   Glucose   63.974076
6  DiabetesPedigreeFunction    0.728482
4                   Insulin    0.030211
2             BloodPressure    0.015793
3             SkinThickness    0.011114


In [29]:
top_kbest = kbest_features["Feature"].head(7)

print("Random Forest Top Features:")
print(list(top_rf))

print("\nSelectKBest Top Features:")
print(list(top_kbest))

Random Forest Top Features:
['BMI', 'Age', 'Pregnancies', 'Glucose', 'BloodPressure', 'DiabetesPedigreeFunction', 'Insulin']

SelectKBest Top Features:
['BMI', 'Age', 'Pregnancies', 'Glucose', 'DiabetesPedigreeFunction', 'Insulin', 'BloodPressure']


# 1. Which features were most important according to Random Forest?

The most important features are ['BMI', 'Age', 'Pregnancies', 'Glucose', 'BloodPressure', 'DiabetesPedigreeFunction', 'Insulin']. These features had the highest importance values, meaning they contributed the most to the model’s decision-making process. Random Forest determines importance by measuring how much each feature reduces impurity across the trees, so higher values indicate stronger predictive power.


# 2. Which features were selected by SelectKBest?

The most important features selected by SelectKBest were ['BMI', 'Age', 'Pregnancies', 'Glucose', 'DiabetesPedigreeFunction', 'Insulin', 'BloodPressure']. 

# 3. Why might feature selection improve a model?

Feature selection can improve a model by removing irrelevant or less useful features, which reduces noise and helps the model focus on the most important information.

In this project, after applying SelectKBest, model performance either improved or remained strong. For example:

Logistic Regression improved from 0.935 to 0.948 accuracy

KNN improved from 0.903 to 0.929 accuracy

Random Forest improved from 0.993 to 1.00 accuracy

This shows that removing less important features helped simplify the model while maintaining or improving performance

# 4. Which model performed best overall?

The Random Forest model performed the best overall.

With all features, it achieved an accuracy of 0.9935, and after feature selection, it reached a perfect accuracy of 1.00. It also had zero false negatives in both cases, meaning it correctly identified all patients with diabetes.

This is especially important in a medical context, where missing a positive case can have serious consequences. Compared to other models, Random Forest consistently delivered the highest accuracy and best overall performance.

# 5. Why is accuracy alone not enough?

Accuracy alone is not enough because it does not fully capture how well a model performs, especially in situations where different types of errors have different consequences.

For example, in this project:

Logistic Regression had high accuracy (0.935) but misclassified 10 non-diabetic cases as diabetic

SVM and KNN also had good accuracy but still produced false negatives and false positives

In medical prediction, false negatives are especially dangerous, because they represent patients who actually have diabetes but are predicted as healthy.

Metrics such as recall, precision, and F1-score, along with the confusion matrix, provide a more complete understanding of model performance. These metrics help evaluate how well the model identifies positive cases and avoids critical errors.